# Extract centerlines
This notebook is to extrack centerlines

In [10]:
#import pckgs
import cv2
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import csv
import re

import os
from natsort import natsorted

from scipy.interpolate import splprep, splev

from skimage.morphology import medial_axis, skeletonize
from skimage import data
from skimage.util import invert
import skimage.graph

import scipy.fftpack
from centerline.src.make_skeleton import make_skeleton

In [74]:
#define h5 to read from:

#old version, filtered
#df=pd.read_hdf('/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/all_hdf5_filtered/2020-07-01_18-36-25_control_worm6-channel-0-bigtiffDLC_resnet50_HeadTailAug10shuffle1_590000_filtered.h5')

#new version, filtered
df=pd.read_hdf('/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/avi_all/2020-07-01_13-21-00_chemotaxisl_worm1-channel-0-bigtiffDLC_resnet50_HeadTailAug10shuffle1_275000_filtered.h5')

#select images to process
path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/Annotation_OT/\
2020-07-01_13-21-00_chemotaxisl_worm1-channel-0-bigtiff_binary_th20'

img_path=os.path.join(path, 'processed/')
# Annotation_OT/2020-06-30_16-25-56_chemotaxis_worm2-channel-0-bigtiff_binary_th20/processed/'
img_list=natsorted(os.listdir(img_path))

scorer=df.columns.get_level_values(0)[0]
head_x=df[scorer]['Head']['x'].values
head_y=df[scorer]['Head']['y'].values
tail_x=df[scorer]['Tail']['x'].values
tail_y=df[scorer]['Tail']['y'].values
scorer=df.columns.get_level_values(0)[0]

num_splines=100
#img=img_all[i]



In [75]:
#to visualize

#create writer object
output_path=os.path.join(path, 'centerlines/')

csvfileX=open(output_path+'spline_X_coords.csv','w', newline='')
csvfileX_writer=csv.writer(csvfileX)
csvfileY=open(output_path+'spline_Y_coords.csv','w', newline='')
csvfileY_writer=csv.writer(csvfileY)

csvfilename=open(output_path+'filename.csv','w', newline='')
csvfilename_writer=csv.writer(csvfilename)




#Compile regular expressions into pattern objects 
regex = re.compile(r'\d+')

#set counters for failed centerlines
c1=0
c2=0

#define expected skeleton minimum length
skel_min_length=300

for filename in img_list:
    if 'img' in filename:
        
        print(filename)
        
        i=int(regex.search(filename).group(0))
        print(i)
        start_point=(int(head_y[i]), int(head_x[i]))
        end_point = (int(tail_y[i]), int(tail_x[i]))
        #print(start_point)
        #print(end_point)
        
        img=tiff.imread(os.path.join(img_path,filename))
        
        annotated_img=img.copy()
        cv2.circle(annotated_img,(int(head_x[i]), int(head_y[i])),10, (150,150,150), 2)
        cv2.circle(annotated_img,(int(tail_x[i]), int(tail_y[i])),10, (150,150,150), 2)
        
#         plt.figure(figsize=(10,10), dpi=150)
#         plt.subplot(1,2,1)
#         plt.imshow(annotated_img)
      
        u, skel_coords, spline_coords, K=make_skeleton(start_point, end_point,num_splines, img)
        #print(skel_coords[0].shape)
        
        #try to unpack the skel_coords as an array of int (will fail if skel_coords contains NaN)
        try: coords=np.asarray(list(zip(*skel_coords)), dtype=int)# the dtype=int makes sures no NaN can be passed, which will cause trouble when indexing
        except:
            c1=c1+1
            #print('head and tail were too close, no centerline could be created', c1)
            
        else:
            #counter for short skel coords
            if len(skel_coords[0])<=skel_min_length:
                c2=c2+1
                #print('short centerline', c2)
            
            #draw zeros on the skel_img where the are coordinates
            skel_img=img.copy()
            for idx,coords in enumerate(coords):
                skel_img[coords[0],coords[1]]=0
            
            #save coordinates of the good centerlines
            if len(skel_coords[0])>skel_min_length:
                x,y=np.asarray(spline_coords)
                csvfileX_writer.writerow(x)
                csvfileY_writer.writerow(y)
                csvfilename_writer.writerow(filename)
            #plotting
#             plt.figure(figsize=(10,10), dpi=150)
#             plt.subplot(1,2,1)
#             plt.imshow(annotated_img)
            #plt.subplot(1,3,1)
            #plt.imshow(img)
#             plt.subplot(1,2,2)
#             plt.imshow(skel_img)
#             plt.show()
            #check if the skel_coords are too short
csvfileX.close()
csvfileY.close()
csvfilename.close()
print('Failed centerlines: ', c1,'\nVery short centerlines:', c2, '\nGood centerlines: ', len(img_list)-c1-c2,'\nTotal centerlines: ', len(img_list))

img000000.tif
0
img000078.tif
78
img010324.tif
10324
img010437.tif
10437
img010804.tif
10804
img070070.tif
70070
img070160.tif
70160
img070238.tif
70238
img070422.tif
70422
img070593.tif
70593
img071126.tif
71126
img118293.tif
118293
img118307.tif
118307
img118374.tif
118374
img118411.tif
118411
img118457.tif
118457
img118480.tif
118480
img118509.tif
118509
img118544.tif
118544
img118660.tif
118660
img118712.tif
118712
img118777.tif
118777
Failed centerlines:  0 
Very short centerlines: 6 
Good centerlines:  16 
Total centerlines:  22


In [49]:
x,y=np.asarray(spline_coords)

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
output_path=os.path.join('/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/skeleton_new/',re.split('-channel',re.split('/',input_filename)[-1])[0])


csvfileK=open(output_path+'_spline_K.csv','w', newline='')
csv_writerK=csv.writer(csvfileK)


csv_writerK.writerow(K)

In [42]:
bodyparts=[]
for i in np.arange(100):
    bodyparts.append('bodypart'+str(i))

In [43]:
bodyparts

['bodypart0',
 'bodypart1',
 'bodypart2',
 'bodypart3',
 'bodypart4',
 'bodypart5',
 'bodypart6',
 'bodypart7',
 'bodypart8',
 'bodypart9',
 'bodypart10',
 'bodypart11',
 'bodypart12',
 'bodypart13',
 'bodypart14',
 'bodypart15',
 'bodypart16',
 'bodypart17',
 'bodypart18',
 'bodypart19',
 'bodypart20',
 'bodypart21',
 'bodypart22',
 'bodypart23',
 'bodypart24',
 'bodypart25',
 'bodypart26',
 'bodypart27',
 'bodypart28',
 'bodypart29',
 'bodypart30',
 'bodypart31',
 'bodypart32',
 'bodypart33',
 'bodypart34',
 'bodypart35',
 'bodypart36',
 'bodypart37',
 'bodypart38',
 'bodypart39',
 'bodypart40',
 'bodypart41',
 'bodypart42',
 'bodypart43',
 'bodypart44',
 'bodypart45',
 'bodypart46',
 'bodypart47',
 'bodypart48',
 'bodypart49',
 'bodypart50',
 'bodypart51',
 'bodypart52',
 'bodypart53',
 'bodypart54',
 'bodypart55',
 'bodypart56',
 'bodypart57',
 'bodypart58',
 'bodypart59',
 'bodypart60',
 'bodypart61',
 'bodypart62',
 'bodypart63',
 'bodypart64',
 'bodypart65',
 'bodypart66',
 'bod

In [47]:
#create pandas dataframe
scorer=['Ulises']
bodyparts=[]
for i in np.arange(100):
    bodyparts.append('bodypart'+str(i))
coords=['x','y']
arrays=pd.MultiIndex.from_product([scorer,bodyparts, coords],

                           names=['scorer','bodyparts', 'coords'])
frames=np.arange(0,10)
skeleton_df=pd.DataFrame(index=frames, columns=arrays)

In [48]:
skeleton_df.head(2)

scorer       Ulises                                                    \
bodyparts bodypart0      bodypart1      bodypart2      bodypart3        
coords            x    y         x    y         x    y         x    y   
0               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
1               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   

scorer                    ...                                                  \
bodyparts bodypart4       ... bodypart95      bodypart96      bodypart97        
coords            x    y  ...          x    y          x    y          x    y   
0               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
1               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   

scorer                                     
bodyparts bodypart98      bodypart99       
coords             x    y          x    y  
0                NaN  NaN        NaN  NaN  
1                NaN  NaN        NaN  NaN  

[2 rows x 200 columns]

In [89]:
skeleton_df.iloc[0]['Ulises',:]

bodyparts   coords
bodypart0   x           2
            y         NaN
bodypart1   x           3
            y         NaN
bodypart2   x         NaN
                     ... 
bodypart97  y         NaN
bodypart98  x         NaN
            y         NaN
bodypart99  x         NaN
            y         NaN
Name: 0, Length: 200, dtype: object

In [79]:
skeleton_df

scorer       Ulises                                                    \
bodyparts bodypart0      bodypart1      bodypart2      bodypart3        
coords            x    y         x    y         x    y         x    y   
0                 2  NaN         3  NaN       NaN  NaN       NaN  NaN   
1               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
2               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
3               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
4               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
5               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
6               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
7               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
8               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   
9               NaN  NaN       NaN  NaN       NaN  NaN       NaN  NaN   

scorer                    ...                                                  \
bodyparts bodypart4       ... bodypart95      bodypart96      bodypart97        
coords            x    y  ...          x    y          x    y          x    y   
0               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
1               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
2               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
3               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
4               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
5               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
6               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
7               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
8               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   
9               NaN  NaN  ...        NaN  NaN        NaN  NaN        NaN  NaN   

scorer                                     
bodyparts bodypart98      bodypart99       
coords             x    y          x    y  
0                NaN  NaN        NaN  NaN  
1                NaN  NaN        NaN  NaN  
2                NaN  NaN        NaN  NaN  
3                NaN  NaN        NaN  NaN  
4                NaN  NaN        NaN  NaN  
5                NaN  NaN        NaN  NaN  
6                NaN  NaN        NaN  NaN  
7                NaN  NaN        NaN  NaN  
8                NaN  NaN        NaN  NaN  
9                NaN  NaN        NaN  NaN  

[10 rows x 200 columns]

In [59]:
spline_coords[0]

array([272.06519468, 272.58295585, 273.83594757, 275.45451869,
       277.18062054, 278.85271699, 280.39069441, 281.78077162,
       283.06040993, 284.30322311, 285.60388737, 287.06305135,
       288.77224611, 290.79879811, 293.17199568, 295.87981947,
       298.8777941 , 302.09985613, 305.46923557, 308.90933739,
       312.35432305, 315.75583285, 319.08361524, 322.32469668,
       325.48252851, 328.57613393, 331.63914107, 334.71647424,
       337.85592825, 341.09801144, 344.46573704, 347.95441434,
       351.52158106, 355.08191195, 358.51730787, 361.69414902,
       364.48091272, 366.76579114, 368.47425226, 369.58272553,
       370.11556497, 370.13109794, 369.70679018, 368.92441036,
       367.85520486, 366.54672517, 365.01994749, 363.27563206,
       361.30182077, 359.08133791, 356.59929018, 353.85016732,
       350.84125007, 347.59146466, 344.12941292, 340.49139539,
       336.71943456, 332.85929798, 328.95852148, 325.06443229,
       321.22217226, 317.47272096, 313.85091893, 310.38

In [12]:
skel_coords

(array([272, 272, 272, 272, 272, 273, 273, 274, 274, 274, 275, 275, 275,
        275, 276, 276, 277, 277, 277, 278, 278, 278, 279, 279, 279, 280,
        280, 280, 281, 281, 281, 282, 282, 282, 283, 283, 283, 284, 284,
        284, 285, 285, 285, 286, 286, 286, 287, 287, 287, 288, 288, 289,
        289, 289, 290, 290, 291, 291, 292, 292, 293, 293, 294, 294, 295,
        295, 296, 296, 297, 298, 298, 299, 299, 300, 301, 301, 302, 303,
        304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 313, 314, 315,
        316, 317, 317, 318, 319, 320, 320, 321, 322, 322, 323, 324, 324,
        325, 326, 327, 327, 328, 328, 329, 330, 330, 331, 332, 332, 333,
        334, 334, 335, 336, 336, 337, 338, 338, 339, 340, 340, 341, 342,
        343, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354,
        355, 356, 357, 358, 358, 359, 360, 360, 361, 362, 362, 363, 363,
        364, 364, 365, 365, 366, 366, 366, 367, 367, 367, 368, 368, 368,
        369, 369, 369, 369, 370, 370, 370, 370, 370